## Previamente en SQL EDITOR - Supabase

**Consideraciones** para hacer en el proyecto de Supabase. En la sección de SQL Editor. <p>
- Crear vistas en public para que apunten a las tablas en el esquema 'raw'.(sql editor)
```
CREATE VIEW public.customers_clean as
SELECT * FROM clean.customers_clean;

CREATE VIEW public.products_clean as
SELECT * FROM clean.products_clean;

CREATE VIEW public.orders_clean as
SELECT * FROM clean.orders_clean;

CREATE VIEW public.orders_products_clean as
SELECT * FROM clean.orders_products_clean;
```

## Librerias

In [4]:
import os
from supabase import create_client, Client
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm


## Funciones Supabase

Función *extract_supabase()* tambien implementada en [pruebas_transform_load]()

In [ ]:
def extract_supabase(endpoint, esquema):
    """
    Extrae datos crudos desde una tabla en Supabase asociada a un endpoint específico.
    
    Características principales:
    - Los datos se obtienen en lotes de 1000 filas (paginación con .range).
    - Por defecto soporta hasta 5000 registros, pero se puede ampliar el rango.
    - Para el endpoint "products", se especifican las columnas exactas a consultar
      (evita traer campos innecesarios o complejos como JSON anidados).
    - Para otros endpoints, se extraen todas las columnas disponibles con '*'.
    - El proceso se detiene automáticamente cuando no existen más filas en el rango.
    - Une todos los lotes extraídos en un único DataFrame de Pandas.
    
    Input: endpoint(str)
    Nombre del endpoint a consultar (ejemplo: "products", "orders", "customers").
    
    Output: Un DataFrame con todos los registros obtenidos de la tabla `{endpoint}_raw`.
    """
    name_table= f"{endpoint}_{esquema}"
    rango= [0,1000,2000,3000,4000]
    # evaluar o almacenar mientras exista rango en la tabla si lanza el error entonces parar

    parts_table=[]

    if esquema=='raw' and endpoint=='products': # columnas obtenidas luego de query en SQL EDITOR
        select_query=' "product.id" , "product.name" , "product.page_title" , "product.description" , "product.meta_description" , "product.price" , "product.cost_per_item" , "product.compare_at_price" , "product.weight" , "product.stock" , "product.stock_unlimited" , "product.stock_threshold" , "product.stock_notification" , "product.sku" , "product.brand" , "product.barcode" , "product.featured" , "product.reviews_enabled" , "product.status" , "product.shipping_required" , "product.type" , "product.days_to_expire" , "product.created_at" , "product.updated_at" , "product.package_format" , "product.length" , "product.width" , "product.height" , "product.diameter" , "product.google_product_category" , "product.images" , "product.variants" , "product.fields" , "product.permalink" , "product.discount" , "product.currency" '
        # select_query=products_column
    else:
        select_query="*"

    # Obtener data de Supabase por partes
    for valor in rango:
        table_chunk = ( 
        supabase.table(name_table)
        .select(select_query)
        .range(valor,valor+999) # 0,999 , lo mismo= rango[ind]
        .execute()
            )
        if len(table_chunk.data): # si existe la tabla en ese rango
            parts_table.append(table_chunk.data)
        else:
            break

    # Juntar todas las listas en una sola lista
    completed_table =[]
    for chunk in parts_table:
        completed_table+= chunk
    
    print(f'✅ Extracción correcta realizada para {name_table}')
    print(f"Filas: {len(completed_table)}\n")

    return pd.DataFrame(completed_table)

## Flujo de Ejecución

In [3]:
load_dotenv()
url= os.environ.get("SUPABASE_URL")
key= os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(url, key)

In [6]:
endpoints= [ "products","customers","orders","orders_products"]
esquema='clean'
df_clean= {}
for endpoint in tqdm(endpoints):
    df_clean[f'{endpoint}_clean']= extract_supabase(endpoint=endpoint, esquema=esquema)

 25%|██▌       | 1/4 [00:01<00:04,  1.64s/it]

✅ Extracción correcta realizada para products_clean
Filas: 1339



 50%|█████     | 2/4 [00:02<00:02,  1.00s/it]

✅ Extracción correcta realizada para customers_clean
Filas: 1851



 75%|███████▌  | 3/4 [00:03<00:01,  1.11s/it]

✅ Extracción correcta realizada para orders_clean
Filas: 2333



100%|██████████| 4/4 [00:04<00:00,  1.03s/it]

✅ Extracción correcta realizada para orders_products_clean
Filas: 2973



In [10]:
df_clean['customers_clean'].columns

Index(['id_cliente', 'cliente_nombre', 'status', 'acepta_marketing',
       'cliente_ciudad', 'cliente_municipalidad', 'cliente_pais',
       'cliente_ciudad_missing', 'cliente_municipalidad_missing',
       'acepta_marketing_missing', 'marketing_cat'],
      dtype='object')

In [11]:
df_clean['orders_clean'].columns

Index(['id_orden', 'fecha_creacion', 'moneda', 'precio_total',
       'estado_cumplimiento', 'empresa_envio', 'id_cliente', 'region_envio',
       'pais_envio', 'municipalidad_envio', 'estado_orden', 'estado_envio',
       'estado_cliente', 'productos_ids'],
      dtype='object')

In [13]:
df_clean['products_clean'].columns

Index(['id_producto', 'descripcion', 'precio', 'stock', 'umbral_stock',
       'notificacion_stock', 'marca', 'reseñas_habilitadas', 'estado',
       'fecha_creacion', 'fecha_actualizacion', 'moneda'],
      dtype='object')

In [12]:
df_clean['orders_products_clean'].columns

Index(['id_orden', 'id_producto'], dtype='object')

In [7]:
df_clean.keys()

dict_keys(['products_clean', 'customers_clean', 'orders_clean', 'orders_products_clean'])

In [9]:
len(df_clean['customers_clean'])

1851

In [ ]:
import streamlit as st
import plotly.express as px
